# Language Detection  

Airbnb attracts tourists from all over the world, which means the reviews come in dozens of languages. Since our NLP pipeline and models are built for English, the first step is to filter out everything else.

langdetect was used to identify the language of each review, analyzing only the first 300 characters per comment, which is enough to detect the language without processing the full text unnecessarily.

In [1]:
#Install one time
#! pip install nltk
#! pip install vaderSentiment
#! pip install pytrends
#! pip install textblob
#! pip install wordcloud
#! pip install gensim
#! pip install seaborn
#! pip install TextBlob
#! pip install joblib
#! pip install tqdm
#! pip install langdetect
#! pip install pandarallel

In [2]:
import nltk
import logging
import numpy as np
import pandas as pd
import seaborn as sns
from textblob import TextBlob
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
import re
import string
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from tqdm.auto import tqdm
tqdm.pandas()
import time
from langdetect import detect
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 6 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


In [3]:
#Download one time NLTK 
#nltk.download('punkt')
#nltk.download('stopwords')
#nltk.download('wordnet')
#nltk.download('averaged_perceptron_tagger_eng')

# Loading and Inspecting dataset

In [4]:
df = pd.read_csv(r"C:\Users\kumri\Downloads\Data Science Notebooks\AirBnB sentiment analysis\reviews.csv")
df.sample(5)

,listing_id,id,date,reviewer_id,reviewer_name,comments
161260,13127944.0,9.317092e+07,8/11/2016,66487797,Vjekoslav,Tom and Vera are nice host. Everthing was as a...
277705,36258613.0,4.832659e+08,7/7/2019,254603676,Corentin,"Appartement très propre, lumineux, et qui corr..."
233350,23977668.0,7.612724e+08,5/23/2021,33252174,Sjoukje,Topverblijf op toplocatie! Zeer complete inric...
297194,46591961.0,5.700680e+17,2/25/2022,133824023,Diogo,one of the best things in our roadtrop was thi...
154283,12920724.0,1.236914e+08,12/30/2016,9894265,Lotte,a beautiful place! nice to have the delicious ...


In [5]:
df.columns

Index(['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments'], dtype='object')

In [6]:
df.shape

(342904, 6)

In [7]:
#This helps allows to see there are null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 342904 entries, 0 to 342903
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   listing_id     342904 non-null  float64
 1   id             342904 non-null  float64
 2   date           342904 non-null  object 
 3   reviewer_id    342904 non-null  int64  
 4   reviewer_name  342904 non-null  object 
 5   comments       342888 non-null  object 
dtypes: float64(2), int64(1), object(3)
memory usage: 15.7+ MB


In [8]:
#Checking if there are duplicated values in the df.
df.duplicated().sum()

np.int64(0)

In [9]:
#removing columns that does not aport usefull information.
df = df.drop(['id','reviewer_id','listing_id'], axis=1)
df.head()

,date,reviewer_name,comments
0,3/30/2009,Lam,Daniel is really cool. The place was nice and ...
1,7/9/2012,Gregory,If you want the authentic Amsterdam houseboat ...
2,7/15/2012,Michael,Unique and luxurious to be sure. I couldn't re...
3,4/24/2009,Alice,Daniel is the most amazing host! His place is ...
4,8/12/2012,Brian,My wife and I recently stopped in Amsterdam fo...


In [10]:
pd.set_option('display.max_colwidth', None)
df['comments'].sample(5)

207304                                         Wir hatten einen sehr schönen Aufenthalt! <br/>Die Wohnung ist sehr zentral gelegen und man kann von dort aus alles zu Fuß erreichen. Es war sehr sauber und wunderschön eingerichtet, wir haben an der Wohnung nicht zu bemängeln. Sauber, modern und gemütlich. <br/>Die Decken sind etwas niedrig, für etwas größere Personen vielleicht schwierig. <br/>Der Check-In sowie Check-Out ohne Probleme und sehr simple. <br/>Wir haben von Mariam und Norbert super Tipps bezüglich Parken und Taxi bekommen! <br/>Würde es immer weiter empfehlen!
310972                                                                                                                                                                                                                                                                                                                                                                                                                               

In [11]:
def detect_language(texto):
    try:
        return detect(str(texto)[:300])
    except:
        return 'unknown'

df['language'] = df['comments'].progress_apply(detect_language)
print(df['language'].value_counts().head(10))

  0%|          | 0/342904 [00:00<?, ?it/s]

language
en         258347
fr          28729
de          20008
nl           9815
es           9425
it           4270
pt           1966
unknown      1121
ru           1099
ro            906
Name: count, dtype: int64


Out of 342,904 reviews, 258,347 were identified as English (roughly 75% of the dataset). The remaining 25% includes French, German, Dutch, Spanish and Italian among others, which reflects international tourism profile.

In [22]:
df_english = df[df['language'] == 'en'].copy()
print('English lines:',len(df_english))
print('Discarted lines',len(df)-len(df_english))

English lines: 258347
Discarted lines 84557


## Saving the dataset

  The balanced dataset is saved as a CSV file to be used in the next notebook for text cleaning.

In [23]:
df_english.to_csv(r"C:\Users\kumri\Downloads\Data Science Notebooks\AirBnB sentiment analysis\airbnb_english.csv",
                 index = False)